## **VALIDATION NB**

### YOLO Model Evaluation: Accuracy and Performance

This notebook evaluates a trained **YOLO** object detection model using standard computer vision metrics.

## Accuracy (mAP)

Model performance is measured using:

- **mAP@50**: IoU = 0.5  
- **mAP@50-95**: COCO metric (more strict)  
- **Precision (P)**  
- **Recall (R)**  

Additionally, **per-class mAP@50** is reported for detailed analysis.

### Inference Performance

We also consider real-time performance:

- **Latency**
- **FPS (Frames Per Second)**

> ⚠️ FPS evaluation is currently in progress.

### Evaluation Setup

- Dataset defined via `.yaml`
- Configurable image size and batch size
- Confidence and IoU thresholds

### Goal

Assess both **detection accuracy** and **real-time feasibility** of the model.

In [1]:
# 1. Instalación de dependencias (Ejecutar solo si es necesario)
# !pip install ultralytics torch torchvision

import torch
from datetime import datetime
from ultralytics import YOLO
import os

# ==========================================
# 2. DETECCIÓN Y CONFIGURACIÓN DE GPU
# ==========================================
def check_device():
    if torch.cuda.is_available():
        n_gpus = torch.cuda.device_count()
        gpu_name = torch.cuda.get_device_name(0)
        print(f"✅ GPU DETECTADA: {gpu_name}")
        print(f"Total de GPUs disponibles: {n_gpus}")
        print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        device = 0  # Usar la primera GPU (ID 0)
    else:
        print("No se detectó GPU. Se usará la CPU (Esto será muy lento).")
        device = 'cpu'
    return device

selected_device = check_device()

✅ GPU DETECTADA: NVIDIA RTX 2000 Ada Generation
Total de GPUs disponibles: 1
VRAM Total: 17.18 GB


In [ ]:
MODEL_PATH = 'INPUT YOLO MODEL PATH'
DATA_YAML  = 'INPUT VALIDATION DATASET DATA.YAML PATH'
IMG_SIZE   = 640   # tamaño de imagen para eval
BATCH_SIZE = 16    # reduce a 8 si hay OOM
CONF_THRES = 0.25  # umbral de confianza
IOU_THRES  = 0.5   # IoU para mAP50
DEVICE     = '0' if __import__('torch').cuda.is_available() else 'cpu'

print(f"Modelo : {MODEL_PATH}")
print(f"Dataset: {DATA_YAML}")
print(f"Device : {DEVICE}")

Carpetas en dataset:
['data.yaml', 'images', 'labels']
Número de imágenes train: 2068


**This section evaluates a trained YOLO model on a validation dataset.**

We compute standard detection metrics:
- mAP@50
- mAP@50-95
- Precision
- Recall

Per-class performance is also reported to analyze detection quality across categories.

In [ ]:
# ── mAP Evaluation ----------------------------

model = YOLO(MODEL_PATH)

results = model.val(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    conf=CONF_THRES,
    iou=IOU_THRES,
    device=DEVICE,
    verbose=True,
    save_json=True,   # guarda resultados en formato COCO
    plots=True,       # genera confusion matrix y curvas P/R
)

print("\n" + "="*50)
print("RESULTADOS mAP")
print("="*50)
print(f"mAP@50        : {results.box.map50:.4f}")
print(f"mAP@50-95     : {results.box.map:.4f}")
print(f"Precision (P) : {results.box.mp:.4f}")
print(f"Recall    (R) : {results.box.mr:.4f}")

# Por clase
print("\nmAP@50 por clase:")
names = model.names
for i, ap in enumerate(results.box.ap50):
    print(f"  {names[i]:20s}: {ap:.4f}")